# Fase 4: Evaluación y Exportación / Inferencia

Objetivo: Calcular el desempeño del modelo sobre el conjunto Test (2024), visualizar las predicciones vs reales y simular una predicción futura.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import xgboost as xgb
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = 'Datasets'


ModuleNotFoundError: No module named 'xgboost'

In [ ]:
print("Cargando Dataset Maestro y el Modelo Final...")
df = pd.read_parquet(os.path.join(DATA_DIR, 'dataset_maestro.parquet'))

for c in ['codigo_articulo', 'provincia_norm', 'Pais']:
    df[c] = df[c].astype('category')

model_path = 'modelo_xgboost.pkl'
if os.path.exists(model_path):
    model = joblib.load(model_path)
    print("Modelo cargado exitosamente.")
else:
    print(f"ATENCIÓN: No se encontró {model_path}. Por favor, ejecuta primero el Notebook 03.")


In [ ]:
print("Calculando Métricas MAPE y RMSE en el conjunto de Tést (2024)...")
df_test = df[df['year_natural'] == 2024].copy()

features = [
    'semana', 'mes', 'Temp_media_semanal', 'Precip_total_semanal', 'Viento_max_semanal',
    'eventos_ciclismo', 'proxy_vehiculos_semana', 'unidades_lag_1', 'unidades_lag_2',
    'unidades_lag_4', 'unidades_roll_4', 'codigo_articulo', 'provincia_norm'
]

X_test = df_test[features]
y_test = df_test['unidades']

preds = model.predict(X_test)
# Evitar predicciones negativas en ventas
preds = np.maximum(0, preds)

df_test['prediccion'] = preds

rmse = np.sqrt(mean_squared_error(y_test, preds))
mape = mean_absolute_percentage_error(y_test, preds)

print(f"RMSE (Root Mean Squared Error): {rmse:.2f}")
print(f"MAPE (Mean Abs Percentage Error): {mape:.2%}")


In [ ]:
print("Generando gráfico de Predicción vs Real para los 5 productos más vendidos...")

# Encontrar los 5 más vendidos en 2024
top5_articulos = df_test.groupby('codigo_articulo')['unidades'].sum().nlargest(5).index.tolist()

for art in top5_articulos:
    df_plot = df_test[df_test['codigo_articulo'] == art].copy()
    # Agregar a nivel semana-global por si hay ventas en multiples provincias
    df_plot_agg = df_plot.groupby(['year', 'semana']).agg({
        'unidades': 'sum',
        'prediccion': 'sum'
    }).reset_index()
    
    # Crear un string temporal para x-axis
    df_plot_agg['week_str'] = df_plot_agg['year'].astype(str) + "-W" + df_plot_agg['semana'].astype(str)
    
    plt.figure(figsize=(12, 4))
    plt.plot(df_plot_agg['week_str'], df_plot_agg['unidades'], marker='o', label='Real')
    plt.plot(df_plot_agg['week_str'], df_plot_agg['prediccion'], marker='x', linestyle='--', label='Predicción')
    
    plt.title(f"Ventas vs Predicción en 2024 - Artículo {art}")
    plt.xlabel("Semana ISO")
    plt.ylabel("Unidades Totales")
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
print("Ejemplo de Inferencia para la próxima semana (Uso empresarial)...")

# Crear una fila dummy para la semana que viene.
# El responsable de compras introduce el clima previsto y los eventos para un articulo.

# Construir un dataframe con la misma estructura (X_test) para inferir:
semana_que_viene = pd.DataFrame([{
    'semana': 45,
    'mes': 11,
    'Temp_media_semanal': 18.5,
    'Precip_total_semanal': 65.0, # Lluvia > 50mm
    'Viento_max_semanal': 20.0,
    'eventos_ciclismo': 0,        # No hay eventos
    'proxy_vehiculos_semana': 1200000,
    'unidades_lag_1': 140,
    'unidades_lag_2': 150,
    'unidades_lag_4': 160,
    'unidades_roll_4': 145,
    'codigo_articulo': top5_articulos[0], # El top 1 vendido
    'provincia_norm': 'MADRID'
}])

for c in ['codigo_articulo', 'provincia_norm']:
    semana_que_viene[c] = semana_que_viene[c].astype('category')
    
# Importante: asegurar de que la categoría exista en el modelo
# Para simplificar forzamos el tipo categorico como en train

pred_futura = model.predict(semana_que_viene[features])
pred_futura = max(0, pred_futura[0])

print(f"Predicción para el artículo {top5_articulos[0]} en la provincia MADRID para la semana que viene con alta lluvia:")
print(f" --> {pred_futura:.0f} Unidades")
